# TTS data: listening check

Listen to clips that `prepare_data.py` drops and keeps, to check its filters before training.
Uses the same filter code and thresholds as the script, and its cached `data/audio_stats.csv`
(run `python prepare_data.py` once first, from `kanbior-tts/`).

What to listen for:
- **Dropped** clips should really be bad: wrong or partial text, long silences, noise.
- **Borderline kept** clips (closest to a threshold) should still be good. If not, tighten that threshold.
- **Top voices**: pick the voice(s) for single-voice training.

In [1]:
import random
import sys
from collections import Counter
from pathlib import Path

from IPython.display import Audio, display

sys.path.insert(0, ".")
from prepare_data import build_parser, drop_reason, load_clips

# Same defaults as the script; override here to try other thresholds, e.g. args.max_rate_ratio = 1.3
args = build_parser().parse_args([])
N = 3        # clips per group
SEED = 0     # change to hear other clips

In [2]:
clips, _, _ = load_clips(args.data_dir, args.out_dir / "audio_stats.csv")
for c in clips:
    c["reason"] = drop_reason(c, args)
kept = [c for c in clips if c["reason"] is None]
dropped = [c for c in clips if c["reason"]]
print(f"{len(kept):,} kept, {len(dropped):,} dropped:", dict(Counter(c["reason"] for c in dropped)))

Measuring audio (8,323 clips with audio)...
7,900 kept, 423 dropped: {'rate mismatch': 316, 'mostly silence': 95, 'noisy': 5, 'clipped': 1, 'duration': 6}


In [3]:
def play(c):
    status = c["reason"] or "kept"
    print(f"{c['id']}  {c['speaker_id']}  {c['duration']:.1f} s  [{status}]  "
          f"SNR {c['snr_db']:.0f} dB  speech {c['speech_frac']:.0%}  rate x{c['rate_ratio']:.2f}")
    print("  ", c["text"])
    display(Audio(str(args.data_dir / c["audio_path"])))

def sample(rows, n=N):
    return random.Random(SEED).sample(rows, min(n, len(rows)))

## 1. Dropped clips, by reason

In [4]:
for reason in sorted({c["reason"] for c in dropped}):
    print(f"\n=== {reason} ===")
    for c in sample([c for c in dropped if c["reason"] == reason]):
        play(c)


=== clipped ===
tr_000900  JML_S074  9.5 s  [clipped]  SNR 33 dB  speech 43%  rate x1.00
   Diallo, un já xɔ spaghetti kpɔ́nwun tántɔn nu, bo ná xɔ lait caillé ɖésú ɖó é jí.



=== duration ===
tr_002629  JML_S065  15.1 s  [duration]  SNR 45 dB  speech 44%  rate x0.73
   A nǎ sixú sɛ́ caki gban nú nɔví ce súnnuví ɔ́ nú mì a? Alǒkan tɔn ɖíe: ayizɛ́n vɔ̌, ɖokpó ɖokpó, ko nukún we.


tr_008150  JML_S153  15.0 s  [duration]  SNR 76 dB  speech 68%  rate x0.94
   Azǎn nabí ɖíe ɔ́, un mɔ kwɛ́ kpɛɖé bɔ é jló mì bónú má xɔ mɔ̌to ɖokpó nú asi ce. Lo ɔ́, ée un ma ka tunwun nǔɖé ǎ, un kplá nɔví ce bɔ é kpɔ́n xá mì.


tr_001096  JML_S054  15.5 s  [duration]  SNR 45 dB  speech 57%  rate x0.89
   Hotel élɔ́ ɔ́ jɛ acɔ́! Fí ɖě e è nɔ yí mɛ lɛ́ɛ ɖé ɔ nyɔ́ kpɔ́n. Yě zě azinkpo ɖagbe ɖagbe, tɛnmɛ tɛnmɛ lɛ́ɛ ɖó fí ɔ́, kpódó atín ɖagbe ɖagbe lɛ́ɛ kpó.



=== mostly silence ===
tr_004023  JML_S319  6.2 s  [mostly silence]  SNR 62 dB  speech 37%  rate x0.64
   Akwɛ́ caki ɖokpó gbe ɖokpó.


tr_004215  JML_S033  7.1 s  [mostly silence]  SNR 52 dB  speech 34%  rate x0.55
   Akwɛ́sɛxwétɛn 'UBA


... (3 outputs left out)

tr_000418  JML_S082  7.9 s  [noisy]  SNR 25 dB  speech 58%  rate x1.20
   Vodúnxwe ɖu wɛ mǐ ɖě ɖo to mǐtɔn mɛ égbé nugbǒ. Kpɔ́n vodúnsí élɔ lɛ́ɛ bǐ wɛ tɔ́n mɛ nya.



=== rate mismatch ===
tr_004662  JML_S517  8.4 s  [rate mismatch]  SNR 88 dB  speech 56%  rate x0.64
   Aligbó tɛ́ lɛ́ɛ ka ɖe bɔ è ná gbɔn bó ná dó sɔ́ akwɛ́ dó alǒkan jí?


tr_005087  JML_S549  6.9 s  [rate mismatch]  SNR 90 dB  speech 70%  rate x0.50
   Alɔ̌tlɔ ɔ́ ɖo xasun ɔ́ jí ɖo kéze mɛ.


tr_000533  JML_S005  5.1 s  [rate mismatch]  SNR 50 dB  speech 69%  rate x1.66
   Restaurant ɖokpó mɛ restaurant énɛ́ ɔ́ mɛ nyɔ́ ɖɛkpɛ tawun azinkpo lɛ́ɛ ɖo...


## 2. Borderline kept clips

The kept clips closest to each threshold. `rate x` far from 1 means the speaker talks unusually
fast or slow for that text: often a sign of missing or extra words.

In [5]:
borderline = {
    "lowest speech share": sorted(kept, key=lambda c: c["speech_frac"]),
    "lowest SNR": sorted(kept, key=lambda c: c["snr_db"]),
    "fastest for its text": sorted(kept, key=lambda c: -c["rate_ratio"]),
    "slowest for its text": sorted(kept, key=lambda c: c["rate_ratio"]),
}
for name, rows in borderline.items():
    print(f"\n=== {name} ===")
    for c in rows[:N]:
        play(c)


=== lowest speech share ===
tr_007414  JML_S589  5.6 s  [kept]  SNR 73 dB  speech 40%  rate x1.00
   Axi é é nyí dantɔkpá tɔn ɔ́, é wɛ.


tr_007024  JML_S579  5.3 s  [kept]  SNR 68 dB  speech 40%  rate x0.96
   Avɔ klwítɔ́ élɔ́ lɛ́ɛ tɔn nyɔ́ kpɔ́n tawun wɛ mɛ.


tr_004439  JML_S301  5.8 s  [kept]  SNR 73 dB  speech 40%  rate x1.33
   Ali lɛ́ɛ bǐ xwíí wɛ nɔ́ gɔ́ gbadanu lɛ́ɛ.



=== lowest SNR ===
tr_007369  JML_S082  8.5 s  [kept]  SNR 25 dB  speech 63%  rate x1.02
   Axi yɔyɔ̌ élɔ́ e mɛ un byɔ́ din e ɔ́, fítɛ́ un ka na mɔ nǔ e ba wɛ un ɖe e ɖe bá xɔ kléwún.


tr_003452  JML_S082  6.5 s  [kept]  SNR 25 dB  speech 56%  rate x0.70
   Agbaza cé nɔ híhí, é e un ɖyɔ́ aɖǐ fó gúdo é.


tr_007395  JML_S082  7.9 s  [kept]  SNR 26 dB  speech 56%  rate x0.87
   Axi zǎnmɛ tɔn ɔ́ nɔ́ bɛ́ jijɛ bɛ́sín gan tɛ́nwe mɛ gbadanu vɛlɛɛ.


... (2 outputs left out)

tr_006716  JML_S521  6.7 s  [kept]  SNR 81 dB  speech 82%  rate x1.50
   Atínkɛ́nkwínsatɔ́ ɔ́ ɖɔ nú Mireille ɖɔ é ní má zɛ atínkɛ́nkwín para we wú gbe ɖokpó ó.



=== slowest for its text ===
tr_004163  JML_S579  5.4 s  [kept]  SNR 74 dB  speech 42%  rate x0.67
   Akwɛ́ ɔ́ sín vlɔ̌ ɔ́ dǎwe élɔ tɔn wɛ.


tr_001135  JML_S474  5.6 s  [kept]  SNR 73 dB  speech 47%  rate x0.67
   Japon sín asyá to tɔn.


tr_003362  JML_S241  6.1 s  [kept]  SNR 61 dB  speech 64%  rate x0.67
   Agbadékún ɔ́, zǎnzǎn tɛɛn wɛ un nɔ xyá.


## 3. Random kept clips

In [6]:
for c in sample(kept, 2 * N):
    play(c)

tr_007341  JML_S580  5.8 s  [kept]  SNR 49 dB  speech 64%  rate x1.09
   Axi yaɖé yaɖé e acɛkpikpa Benɛ́ɛ to ɔ́ tɔn gbá lɛ́ɛ é ɖokpó.


tr_003367  JML_S111  10.2 s  [kept]  SNR 68 dB  speech 69%  rate x0.93
   Agban bɛ́ ɖó kɛkɛ́ ji zɛ jlɛ wu nyí nǔ ɖagbe ɖé ǎ. É wɛ nɔ hɛn alitawóvi wa.


tr_006600  JML_S517  6.2 s  [kept]  SNR 53 dB  speech 69%  rate x1.28
   Atínkwínsínsɛ́n élɔ ɔ́, ée è nɔ ɖu ɔ́ é wɛ bɔ é ká ɖiga bo ɖo gblɛlɛ bɔ yě kplé sukpɔ́ tɔn ɖó fí.


tr_007729  JML_S504  6.3 s  [kept]  SNR 70 dB  speech 70%  rate x0.98
   Aximɛ ɖíe ɖo fí e bɔ nǎjinɔ mǐtɔn jínjɔ́n ayǐ bó ɖo hweví sa wɛ.


tr_003683  JML_S521  5.4 s  [kept]  SNR 88 dB  speech 56%  rate x0.71
   Ahannutɔ́ wɛ bó nɔ nu bǐya tawun.


tr_000347  JML_S455  5.9 s  [kept]  SNR 59 dB  speech 47%  rate x0.73
   Nyɛ ɔ́ un tunwun ɖɔ hwɛ ná bló hú mɔ̌ mɛ.


## 4. Top voices (most clean hours in the downloaded audio)

In [7]:
hours = Counter()
for c in kept:
    hours[c["speaker_id"]] += c["duration"] / 3600
for speaker, h in hours.most_common(5):
    print(f"\n=== {speaker}: {h:.2f} h clean ===")
    for c in sample([c for c in kept if c["speaker_id"] == speaker], 2):
        play(c)


=== JML_S082: 0.64 h clean ===
tr_006301  JML_S082  8.2 s  [kept]  SNR 46 dB  speech 71%  rate x1.20
   Asán xixo kplɔ́n wɛ mɛ élɔ lɛ́ɛ ka ɖe a? Amɔ̌ nǔ é myá nukún nú mɛ é, è nɔ lébo d'éwú titewungbe.


tr_006658  JML_S082  8.9 s  [kept]  SNR 61 dB  speech 67%  rate x1.20
   Atínkɛnfúnnú élɔ ɔ́ é bɔ tawun wɛ mɛ, un ɖo tagba ɖěbǔ jɛ́ wɛ cóbó ɖo atínkɛn ɔ́ bǐ fún wɛ kpowun ǎ.



=== JML_S041: 0.48 h clean ===
tr_008230  JML_S041  8.4 s  [kept]  SNR 68 dB  speech 64%  rate x0.97
   Azǐin sín hwenu wɛ mǐ ɖe. Bianca yi aximɛ bo hɛn azǐin gěgé wá xwé gbe.


tr_002234  JML_S041  6.1 s  [kept]  SNR 65 dB  speech 64%  rate x0.97
   A ka ko ɖu nǔ din a? Un jló ná sɛ́ nǔɖuɖu ɖokpó sɛ́dó we.



=== JML_S111: 0.45 h clean ===
tr_005776  JML_S111  13.1 s  [kept]  SNR 63 dB  speech 76%  rate x1.11
   Asyá to ɖokpó ɖokpó tɔn wɛ nyí nǔsísí. Asyá to ɖé lɛ́ɛ tɔn ka tíin bo nɔ nyí sinmɛ atɔnnɔ bɔ ɖiɖe ɖé nɔ gɔ́ ná ǎ.


tr_005788  JML_S111  13.6 s  [kept]  SNR 62 dB  speech 75%  rate x1.12
   Asyá to ɖé lɛ́ɛ tɔn nɔ tíin bó nɔ́ ɖó sinmɛ wunmɛ wunmɛ lɛ́ɛ bɔ a nǎ mɔ xwinxwín ɖé lɛ́ɛ bɔ è nɔ zɛ ɖ'é mɛ bɔ é nɔ́ nyɔ́ káká.



=== JML_S104: 0.38 h clean ===
tr_003949  JML_S104  6.9 s  [kept]  SNR 48 dB  speech 76%  rate x0.98
   Akpa ɖé è blǎ nú Kɔmlá ɔ́ nɔ́ ɖe nú é ɖi zɔnlin gěgé ɔ́.


tr_004336  JML_S104  10.6 s  [kept]  SNR 46 dB  speech 76%  rate x1.06
   Akófá jɛ sísɔ́sísɔ́ jí ɖě zozo tɔn yi jǐ ɖo sɔ e wáyi ɔ́ ɖo gan tɛ́nnɛ gbadanu tɔn mɛ ɔ́.



=== JML_S580: 0.37 h clean ===
tr_004974  JML_S580  8.5 s  [kept]  SNR 58 dB  speech 62%  rate x1.08
   Alɔnulókan ce wɛ un zǎn sɔ e wá yí é bó sɛ́ akwɛ́ dó xɔ́ntɔn ce ɖokpó é nyɔ́ ɖésú tawun.


tr_005162  JML_S580  8.0 s  [kept]  SNR 39 dB  speech 66%  rate x0.83
   Ami sísá ée Kɔsí nɔ́ zán é ɔ́, ami vɔvɔ kpódó ama kpó wɛ è blǒ ná.


## Next

If a threshold is wrong, change it with the script's flags, e.g.
`python prepare_data.py --max-rate-ratio 1.3 --min-speech-frac 0.5` (see `python prepare_data.py -h`).